In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tqdm
import os
from source.random_walk import generate_head_direction_walk, toroidal_to_solid_angle
from source.plot_tools import activity_map
from source.attractor_networks.single_bump import ToroidZhang1996

In [ ]:
dt = 0.5e-3

net = ToroidZhang1996(n = 64, dt=dt,revolutions = 1)
net.warm_up()
print(np.mean(net.s),np.max(net.s))
plt.imshow(net.s)

In [ ]:
## Simulation parameters:
n_min_plots = 10
dt = 0.5e-3

net = ToroidZhang1996(n = 64, dt=dt)
net.warm_up()
ncols = np.int64(np.ceil(np.sqrt(n_min_plots)))
nrows = np.int64(np.ceil(n_min_plots / ncols))
n_plots = nrows * ncols
fig, ax = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
ax = ax.flatten()
phi = 0
omega = 1
n_steps = int(2 * np.pi / omega / dt)
records = np.linspace(0, n_steps - 1, n_plots, dtype=int)
plot_counter = 0
time = 0


for step_iter in range(n_steps):
    if step_iter == records[plot_counter]:
        ax[plot_counter].imshow(net.s)
        ax[plot_counter].set_title(f"$\\phi/2\\pi$={phi/(2*np.pi):.2f}")
        plot_counter += 1
    net.step(omega, 0)
    time += dt
    phi += omega * dt


In [ ]:
T = 4000
dt = 0.5e-3
n = 64
save_dir = "simulation_data"

recorded_cells = [(0,0),(32,32)]

In [ ]:


direction, turn_velocity, time =  generate_head_direction_walk(T, dt)
n_steps = time.shape[0]
net = ToroidZhang1996(n = n, dt=dt)
net.warm_up()
recording = np.zeros((n_steps,len(recorded_cells)))
for step_iter in tqdm.tqdm(range(n_steps)):
    recording[step_iter] = np.array([net.s[cell_index] for cell_index in recorded_cells])
    net.step(*turn_velocity[step_iter])


os.makedirs(save_dir, exist_ok=True)
np.save(os.path.join(save_dir,"3d_hd_recording.npy"),recording)
np.save(os.path.join(save_dir,"3d_hd_direction.npy"), direction)
np.save(os.path.join(save_dir,"3d_hd_turn_velocity.npy"), turn_velocity)
np.save(os.path.join(save_dir,"3d_hd_time.npy"),time)


In [ ]:
azimuth_activity_map, edges = activity_map(direction[:,1], recording[:,0],nbins=25)
plt.stairs(values = azimuth_activity_map,edges = edges[0])

In [ ]:
recording = np.load(os.path.join(save_dir,"3d_hd_recording.npy"))
direction = np.load(os.path.join(save_dir,"3d_hd_direction.npy"))
turn_velocity = np.load(os.path.join(save_dir,"3d_hd_turn_velocity.npy"))
time = np.load(os.path.join(save_dir,"3d_hd_time.npy"))